# Compare: organizers' evaluation vs. our CV harness

Same model as the organizers (AlexNet `features.3`, 8 frames averaged, scaler -> PCA 95% -> scaler -> RidgeCV), measured three ways:

| | trained on | scored on | |
|---|---|---|---|
| **A** | all training trials | the **same** training trials (968-way ranking, as in notebook 01) | organizers' "78%" — inflated |
| **B** | all training trials | the 102 **hidden test** trials (needs labels: sub-01..04) | the real accuracy |
| **C** | 4/5 of training trials | the held-out 1/5, groups of 102 (our harness) | estimate of B without test labels |

If the harness works, **C should be close to B**, and A should be far above both.

Chance level: top-1 = 1/102 = 0.98%, top-5 = 4.9%, median rank = 51.5.

In [ ]:
import os, sys, glob
import numpy as np
import cv2

sys.path.insert(0, os.path.abspath('..'))          # repo root, so `src` is importable
from src.harness import (DATA_DIR, load_train, load_test, corr_similarity,
                         score, cross_validate, ridge_baseline)

SUBJECTS = [1]       # add 2, 3, 4 (have test labels) or 5, 6 (A and C only)
LAYER    = 'features.3'
N_FRAMES = 8

# frames / features are cached in the (gitignored) data folder, same names as the organizers' notebooks
FRAME_DIR, FEAT_DIR           = f'{DATA_DIR}/frames',      f'{DATA_DIR}/features'
FRAME_TEST_DIR, FEAT_TEST_DIR = f'{DATA_DIR}/frames_test', f'{DATA_DIR}/features_test'

## 1. AlexNet features (organizers' code, run once; all subjects share the same videos)
Videos are looked up by name: train `0001.mp4 .. 1000.mp4`, test `1001.mp4 .. 1102.mp4`.

In [ ]:
def video_path(sid):
    return (f'{DATA_DIR}/stimuli_train/{sid:04d}.mp4' if sid <= 1000
            else f'{DATA_DIR}/stimuli_test/{sid}.mp4')

def dump_frames(stim_ids, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        cap = cv2.VideoCapture(video_path(sid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

trn_vid_ids = list(range(1, 1001))
tst_ids     = list(range(1001, 1103))
dump_frames(trn_vid_ids, FRAME_DIR)
dump_frames(tst_ids, FRAME_TEST_DIR)
print(len(glob.glob(f'{FRAME_DIR}/*.jpg')), 'train frames,', len(glob.glob(f'{FRAME_TEST_DIR}/*.jpg')), 'test frames')

In [ ]:
from net2brain.feature_extraction import FeatureExtractor

for frame_dir, feat_dir in [(FRAME_DIR, FEAT_DIR), (FRAME_TEST_DIR, FEAT_TEST_DIR)]:
    if not os.path.exists(f'{feat_dir}/consolidated_{LAYER}.npz'):
        fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')
        fx.extract(data_path=frame_dir, layers_to_extract=[LAYER], save_path=feat_dir)

In [ ]:
def load_features(feat_dir):
    """{stim_id: mean over the 8 frames of the flattened layer output}  (as in the organizers' features_for)"""
    z = np.load(f'{feat_dir}/consolidated_{LAYER}.npz')
    by_sid = {}
    for k in z.files:
        by_sid.setdefault(int(k[:5]), []).append(z[k].flatten())
    return {sid: np.mean(v, axis=0).astype(np.float32) for sid, v in by_sid.items()}

feats = {**load_features(FEAT_DIR), **load_features(FEAT_TEST_DIR)}
X_tst = np.stack([feats[s] for s in tst_ids])
print(len(feats), 'videos with features, dim', X_tst.shape[1])

## 2. Measure A, B, C for each subject

In [ ]:
results = {}
for sub in SUBJECTS:
    ids, Y = load_train(sub)                        # Y: (n_train, 17, 300)
    X = np.stack([feats[s] for s in ids])
    n = len(ids)

    # A: organizers' check -- predict the training trials the model was fitted on, rank among all n
    P_trn = ridge_baseline(X, Y, X)
    A = score(corr_similarity(Y, P_trn), np.arange(n))

    # B: real test -- fit on all training trials, rank the 102 test videos (rows in recording order)
    eeg_tst, labels = load_test(sub)
    B = None
    if labels is not None:
        P_tst = ridge_baseline(X, Y, X_tst)
        B = score(corr_similarity(eeg_tst, P_tst), labels - tst_ids[0])

    # C: our harness -- 5-fold CV on training data only
    C = cross_validate(X, Y, ridge_baseline)

    results[sub] = {'A': A, 'B': B, 'C': C}
    print(f'sub-{sub:02d} done')

In [ ]:
fmt = lambda r: '   n/a (no labels)' if r is None else f"{r['top1']*100:5.1f}% {r['top5']*100:5.1f}% {r['median_rank']:5.1f}"
print(f"{'':8}{'A: train-on-train':>20}{'B: real test':>22}{'C: harness CV':>22}")
print(f"{'':8}{'top1  top5  medrk':>20}{'top1  top5  medrk':>22}{'top1  top5  medrk':>22}")
for sub, r in results.items():
    print(f"sub-{sub:02d}  {fmt(r['A']):>20}  {fmt(r['B']):>20}  {fmt(r['C']):>20}")
print(f"{'chance':8}{'(1 in n)':>20}{' 1.0%  4.9%  51.5':>22}{' 1.0%  4.9%  51.5':>22}")

**How to read it**
- **A** is high because the model is scored on trials it memorised — it says nothing about new videos.
- **B** is the number that matters, but each subject has only 102 test trials, so it's noisy (one extra hit = +1%), and we can't use it for sub-05/06.
- **C** uses only training data. If it is close to B, we can trust it to compare future improvements.